# Lección 10: Difracción y Redes de Difracción
### Módulo 3: Fundamentos de Óptica

---

La difracción es el fenómeno que ocurre cuando las ondas de luz encuentran un obstáculo o una abertura y se curvan o desvían a su alrededor, extendiéndose más allá de las predicciones de la óptica geométrica. Este comportamiento, inherente a la naturaleza ondulatoria de la luz, fue explicado inicialmente de manera fenomenológica por Christian Huygens y posteriormente formulado matemáticamente por Augustin-Jean Fresnel y Joseph von Fraunhofer.

En esta lección, estudiaremos formalmente la difracción en diferentes regímenes. Comenzaremos con el principio de Huygens-Fresnel y la distinción cuantitativa entre la difracción de Fresnel (campo cercano) y la de Fraunhofer (campo lejano) utilizando el número de Fresnel. Analizaremos en detalle la difracción de Fraunhofer para rendijas individuales y aberturas rectangulares y circulares (el disco de Airy). Discutiremos las limitaciones fundamentales que impone la difracción en la resolución de sistemas ópticos mediante los criterios de Abbe y Rayleigh. Finalmente, estudiaremos las redes de difracción, su principio de funcionamiento por transmisión y reflexión, y su crucial aplicación en la dispersión y espectroscopía cromática de precisión.

---


## Objetivos de Aprendizaje

Al finalizar esta lección, serás capaz de:
1. **Comprender** el principio de Huygens-Fresnel y su papel en la explicación del comportamiento de las ondas al encontrar obstáculos.
2. **Distinguir** cuantitativamente entre los regímenes de difracción de Fresnel (campo cercano) y Fraunhofer (campo lejano) usando el número de Fresnel ($F$).
3. **Derivar** y simular el patrón de intensidad para la difracción por una rendija de ancho finito y por una abertura rectangular.
4. **Caracterizar** la difracción circular de Airy y aplicar el criterio de Rayleigh para determinar el límite de resolución angular de un sistema óptico.
5. **Analizar** el comportamiento físico de las redes de difracción y modelar cómo su poder dispersivo permite separar longitudes de onda individuales.
6. **Validar** analíticamente las aserciones físicas de los mínimos de difracción y los límites de resolución mediante Python.


## 1. Concepto de Difracción y Principio de Huygens

El concepto de propagación rectilínea de la luz es una aproximación válida únicamente cuando el tamaño de los obstáculos o aberturas con los que interactúa la luz es mucho mayor que la longitud de onda ($\lambda$). Cuando las dimensiones son comparables a $\lambda$, la luz se desvía de su trayectoria rectilínea y entra en la región de sombra geométrica. Este fenómeno es la **difracción**.

### 1.1 El Principio de Huygens-Fresnel
El principio de Huygens (1678) establece que:
* *Cada punto de un frente de onda incidente actúa como una fuente secundaria de ondas esféricas (ondas secundarias de Huygens) que se propagan con la misma velocidad y frecuencia que la onda original.*
* *El nuevo frente de onda en un instante posterior es la superficie envolvente de estas ondas secundarias.*

Fresnel (1818) completó este principio añadiendo el concepto de superposición ondulatoria: las ondas secundarias emitidas por cada punto del frente de onda interfieren entre sí de forma constructiva o destructiva. La amplitud y fase en cualquier punto posterior es el resultado de la integración de estas contribuciones coherentes.

### 1.2 ¿Difracción o Interferencia?
Aunque conceptualmente se enseñan como temas separados, no existe una diferencia física fundamental entre ambos. Richard Feynman señaló con humor: *'Nadie ha sido capaz de definir satisfactoriamente la diferencia entre interferencia y difracción. Es sólo una cuestión de uso, y no hay una diferencia física real e importante entre ellos. Lo mejor que podemos hacer es decir que cuando hay sólo unas pocas fuentes (por ejemplo, dos rendijas), el fenómeno se llama interferencia; pero cuando hay un gran número de fuentes (por ejemplo, una rendija continua), se suele llamar difracción'*. En efecto, la difracción se modela matemáticamente como la interferencia de un número infinito de fuentes puntuales infinitesimales distribuidas en el espacio.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Configurar el estilo de graficación científica
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# Parámetros físicos
lam = 633e-9    # Longitud de onda (633 nm - Láser Helio-Neon)
k = 2 * np.pi / lam
a = 3.5e-6      # Ancho de la rendija (3.5 micrometros)
Ns = 60         # Número de fuentes secundarias de Huygens a lo largo de la rendija
x_sources = np.linspace(-a/2, a/2, Ns)

# Grid de observación en el plano x-y
# y representa la distancia de propagación desde la rendija (en metros)
# x representa la coordenada transversal en la pantalla (en metros)
x_grid = np.linspace(-10e-6, 10e-6, 400)
y_grid = np.linspace(0.2e-6, 16e-6, 400)
X, Y = np.meshgrid(x_grid, y_grid)

# Sumar la amplitud compleja de cada onda secundaria cilíndrica
E = np.zeros_like(X, dtype=complex)
for xs in x_sources:
    # Distancia desde la fuente puntual (xs, 0) a cada punto del grid (X, Y)
    r = np.sqrt((X - xs)**2 + Y**2)
    # Onda cilíndrica de Huygens: propaga como exp(i*k*r)/sqrt(r)
    E += np.exp(1j * k * r) / np.sqrt(r)

# Intensidad es proporcional al cuadrado del módulo de la amplitud
Intensity = np.abs(E)**2

plt.figure(figsize=(10, 6.5))
plt.imshow(Intensity, extent=[-10, 10, 0.2, 16], cmap='hot', origin='lower', aspect='auto')
plt.title(r'Propagación de Ondas de Huygens Detrás de una Rendija ($a \approx 5.5\lambda$)', fontsize=13, fontweight='bold')
plt.xlabel(r'Posición Transversal $x$ (\mu m)', fontsize=11)
plt.ylabel(r'Distancia de Propagación $y$ (\mu m)', fontsize=11)
plt.colorbar(label='Intensidad de Luz (u.a.)')
plt.grid(False)  # Desactivar rejilla sobre la imagen
plt.show()


## 2. Difracción de Fresnel vs de Fraunhofer

Dependiendo de la distancia entre la abertura difractora y el plano de observación, el fenómeno se clasifica en dos regímenes bien diferenciados:

### 2.1 El Número de Fresnel ($F$)
Para caracterizar cuantitativamente los límites del campo cercano y lejano, se define el **número de Fresnel ($F$)** como:
$$F = \frac{a^2}{L\lambda}$$
Donde:
* $a$ es la dimensión característica de la abertura (por ejemplo, el ancho de la rendija).
* $L$ es la distancia de la abertura a la pantalla de observación.
* $\lambda$ es la longitud de onda de la luz.

### 2.2 Régimen de Fresnel (Campo Cercano): $F \ge 1$
Ocurre a distancias cortas de la abertura o cuando esta es relativamente grande. Los frentes de onda que llegan a la pantalla son curvos. La difracción de Fresnel requiere aproximaciones más complejas debido a que la fase no es lineal en la abertura. El patrón de difracción muestra oscilaciones rápidas y sigue vagamente la forma geométrica de la rendija.

### 2.3 Régimen de Fraunhofer (Campo Lejano): $F \ll 1$
Ocurre a distancias muy grandes de la abertura (campo lejano) o cuando la luz incidente está colimada (frentes de onda planos). En este régimen, los rayos de luz provenientes de la abertura a un punto de la pantalla son prácticamente paralelos. La intensidad en la pantalla es proporcional a la Transformada de Fourier espacial de la abertura.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parámetros
lam = 633e-9        # Wavelength (633 nm)
a = 50e-6           # Ancho de la rendija (50 micrometros)
k = 2 * np.pi / lam

# Coordenadas en la pantalla (de -90 a 90 micrometros)
x_scr = np.linspace(-90e-6, 90e-6, 1000)

# Definir tres distancias de observación L
# 1. L = 0.05 mm -> F = a^2 / (L * lam) = 79.0 (Campo muy cercano)
# 2. L = 0.6 mm  -> F = 6.58 (Fresnel intermedio)
# 3. L = 15.0 mm -> F = 0.26 (Transición a Fraunhofer)
L_list = [0.05e-3, 0.6e-3, 15e-3]
labels = [
    r'Campo Cercano (Fresnel, $L=0.05$ mm, $F \approx 79.0$)',
    r'Campo Intermedio (Fresnel, $L=0.6$ mm, $F \approx 6.6$)',
    r'Campo Lejano (Fraunhofer, $L=15.0$ mm, $F \approx 0.26$)'
]
colors = ['royalblue', 'forestgreen', 'crimson']

fig, axes = plt.subplots(3, 1, figsize=(11, 8.5), sharex=True)

# Segmentación fina de la rendija para evaluar la integral de difracción (250 puntos)
x_prime = np.linspace(-a/2, a/2, 250)
dx = x_prime[1] - x_prime[0]

for idx, L in enumerate(L_list):
    I_profile = []
    for x in x_scr:
        # Calcular la distancia r desde cada segmento de la rendija (x_prime, 0) al punto de pantalla (x, L)
        r = np.sqrt((x - x_prime)**2 + L**2)
        # Evaluar numéricamente la integral de Rayleigh-Sommerfeld simplificada
        E_field = np.sum(np.exp(1j * k * r) / r) * dx
        I_profile.append(np.abs(E_field)**2)
    
    # Normalizar la intensidad
    I_profile = np.array(I_profile)
    I_norm = I_profile / np.max(I_profile)
    
    # Trazar el perfil
    axes[idx].plot(x_scr * 1e6, I_norm, color=colors[idx], linewidth=2.2, label=labels[idx])
    axes[idx].set_ylabel('Intensidad Norm.', fontsize=10)
    axes[idx].set_ylim(-0.05, 1.05)
    axes[idx].legend(loc='upper right', frameon=True)
    axes[idx].grid(True, linestyle=':', alpha=0.5)
    
    # Dibujar las líneas de la proyección geométrica de los bordes de la rendija (x = +- 25 um)
    axes[idx].axvline(-a*1e6/2, color='gray', linestyle='--', alpha=0.7)
    axes[idx].axvline(a*1e6/2, color='gray', linestyle='--', alpha=0.7)

axes[-1].set_xlabel(r'Posición en la pantalla $x$ (\mu m)', fontsize=11)
plt.suptitle('Patrones de Difracción: Evolución Numérica del Campo Cercano al Lejano', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


## 3. Difracción de Fraunhofer por Rendija Única y Rectangular

En el régimen de campo lejano, la difracción se puede simplificar analíticamente. 

### 3.1 Rendija Única de Ancho Finito $a$
Si consideramos una sola rendija vertical infinita de ancho $a$ a lo largo de la coordenada $x$: la diferencia de camino óptico entre el rayo del centro y un rayo a una distancia $x'$ del centro es $x' \sin\theta$. Integrando la amplitud compleja a lo largo del ancho de la rendija de $-a/2$ a $a/2$, obtenemos la intensidad:
$$I(\theta) = I_0 \operatorname{sinc}^2(\beta) = I_0 \left[ \frac{\sin(\beta)}{\beta} \right]^2$$
Donde el parámetro de difracción $\beta$ es:
$$\beta = \frac{\pi a \sin\theta}{\lambda} \approx \frac{\pi a y}{\lambda L}$$
Los **mínimos de intensidad** se encuentran cuando el numerador $\sin(\beta) = 0$ (excluyendo $\beta = 0$, que es el máximo central):
$$\beta = m\pi \implies a \sin\theta_m = m\lambda, \quad m \in \{\pm 1, \pm 2, \pm 3, \dots\}$$

### 3.2 Abertura Rectangular de Lados $a$ y $b$
Para una abertura rectangular bidimensional de ancho $a$ (eje $x$) y alto $b$ (eje $y$), la intensidad en la pantalla es simplemente el producto de dos patrones de rendija única perpendiculares:
$$I(x, y) = I_0 \operatorname{sinc}^2\left( \frac{\pi a x}{\lambda L} \right) \operatorname{sinc}^2\left( \frac{\pi b y}{\lambda L} \right)$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parámetros de la abertura y del sistema
a = 0.08e-3     # Ancho en x (0.08 mm)
b = 0.16e-3     # Alto en y (0.16 mm)
L = 1.5         # Distancia a la pantalla (1.5 m)
lam = 633e-9    # Longitud de onda del láser (633 nm)

# Coordenadas en la pantalla (de -20 mm a 20 mm)
xs = np.linspace(-20e-3, 20e-3, 500)
ys = np.linspace(-20e-3, 20e-3, 500)
X_scr, Y_scr = np.meshgrid(xs, ys)

# Cálculo de la intensidad de Fraunhofer
# numpy.sinc calcula sin(pi*x)/(pi*x), por lo que dividimos por pi en el argumento
I_rect = (np.sinc(a * X_scr / (lam * L))**2) * (np.sinc(b * Y_scr / (lam * L))**2)

# Aplicar escala de corrección gamma (exponente 0.4) para comprimir el rango dinámico
# Esto permite hacer visibles las franjas secundarias que tienen muy baja intensidad relativa
I_visual = I_rect**0.4

plt.figure(figsize=(9, 8))
plt.imshow(I_visual, extent=[-20, 20, -20, 20], cmap='inferno', origin='lower')
plt.title(f'Patrón de Difracción de Fraunhofer en Apertura Rectangular ($a={a*1e3:.2f}$ mm, $b={b*1e3:.2f}$ mm)', fontsize=12, fontweight='bold')
plt.xlabel('$x$ en la pantalla (mm)', fontsize=11)
plt.ylabel('$y$ en la pantalla (mm)', fontsize=11)
plt.colorbar(label='Intensidad Modificada $I^{0.4}$', shrink=0.8)
plt.grid(False)
plt.tight_layout()
plt.show()


## 4. Discos de Airy y Resolución Óptica

En la mayoría de los instrumentos ópticos (como cámaras, telescopios y el propio ojo humano), las aberturas y lentes son circulares. La difracción en una abertura circular produce un patrón de anillos concéntricos conocido como el **patrón de Airy**.

### 4.1 Apertura Circular de Diámetro $D$
Integrando en coordenadas polares a lo largo de una apertura circular de diámetro $D$, la intensidad angular viene dada por:
$$I(\theta) = I_0 \left[ \frac{2 J_1(u)}{u} \right]^2$$
Donde:
* $J_1(u)$ es la función de Bessel de primer orden y primera especie.
* $u = \frac{\pi D \sin\theta}{\lambda} \approx \frac{\pi D r}{\lambda L}$.

El máximo central brillante se conoce como el **Disco de Airy**. El primer mínimo oscuro de la intensidad ocurre en el primer cero de la función de Bessel $J_1(u)$, el cual se sitúa en $u \approx 3.8317$:
$$\frac{\pi D \sin\theta_{\text{min}}}{\lambda} = 3.8317 \implies \sin\theta_{\text{min}} = 1.2197 \frac{\lambda}{D} \approx 1.22 \frac{\lambda}{D}$$
El radio físico $R_{\text{Airy}}$ del primer anillo oscuro en la pantalla es:
$$R_{\text{Airy}} \approx 1.22 \frac{\lambda L}{D}$$

### 4.2 Límite de Resolución y Criterio de Rayleigh
La difracción impone un límite fundamental a la capacidad de un sistema óptico para separar detalles finos. Si dos objetos luminosos puntuales (por ejemplo, dos estrellas) están extremadamente cerca, sus discos de Airy se solapan en el detector.
* **Criterio de Rayleigh:** Establece que dos imágenes puntuales son justamente resolubles cuando el centro del disco de Airy de una de ellas coincide exactamente con el primer mínimo de difracción de la otra. El ángulo límite de separación es:
  $$\theta_{\text{Rayleigh}} = 1.22 \frac{\lambda}{D}$$
  Si la separación angular entre los objetos es menor que $\theta_{\text{Rayleigh}}$, el sistema óptico los observará como un solo objeto borroso.
* **Límite de Difracción de Abbe:** Ernst Abbe descubrió que para microscopios con lentes de apertura numérica $\text{NA} = n \sin\alpha$, la mínima distancia resoluble en el objeto es $d_{\text{Abbe}} = \frac{\lambda}{2 \text{NA}}$.


In [ ]:
from scipy.special import j1
import numpy as np
import matplotlib.pyplot as plt

def airy_profile(x, D, lam, L):
    # u = pi * D * x / (lambda * L)
    u = np.pi * D * x / (lam * L)
    # Evitar indeterminación por división por cero
    u = np.where(u == 0, 1e-15, u)
    return (2 * j1(u) / u)**2

# Parámetros ópticos típicos
D = 0.4e-3       # Diámetro de la abertura circular (0.4 mm)
lam = 550e-9     # Luz verde (550 nm)
L = 1.8          # Distancia a la pantalla (1.8 m)

# Separación angular y lineal teórica según el criterio de Rayleigh
theta_Rayleigh = 1.22 * lam / D
x_Rayleigh = theta_Rayleigh * L   # Separación lineal en pantalla (m)

# Pantalla lineal (coordenada transversal x de -7 mm a 7 mm)
x = np.linspace(-7e-3, 7e-3, 1000)

# Tres separaciones físicas a simular
separaciones = [0.65 * x_Rayleigh, 1.0 * x_Rayleigh, 1.65 * x_Rayleigh]
titles = [
    'No Resueltas ($d = 0.65 d_R$)',
    'Límite de Rayleigh ($d = 1.0 d_R$)',
    'Totalmente Resueltas ($d = 1.65 d_R$)'
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for idx, sep in enumerate(separaciones):
    # Intensidad individual de las fuentes desplazadas
    I1 = airy_profile(x - sep/2, D, lam, L)
    I2 = airy_profile(x + sep/2, D, lam, L)
    I_total = I1 + I2
    
    axes[idx].plot(x * 1e3, I1, 'g--', linewidth=1.5, alpha=0.65, label='Fuente 1')
    axes[idx].plot(x * 1e3, I2, 'b--', linewidth=1.5, alpha=0.65, label='Fuente 2')
    axes[idx].plot(x * 1e3, I_total, 'r-', linewidth=2.2, label='Intensidad Total')
    
    axes[idx].set_title(titles[idx], fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Posición en pantalla $x$ (mm)', fontsize=10)
    axes[idx].set_xlim(-6, 6)
    axes[idx].set_ylim(-0.05, 2.1)
    axes[idx].grid(True, linestyle=':', alpha=0.5)
    
    # En el límite de Rayleigh, marcar el valle característico (~73.5% del pico máximo individual)
    if idx == 1:
        axes[idx].axhline(0.735, color='gray', linestyle=':', label='Mínimo de Rayleigh (~73.5%)')
        
    if idx == 0:
        axes[idx].set_ylabel('Intensidad Normalizada', fontsize=11)
        axes[idx].legend(loc='upper right', frameon=True)

plt.suptitle('Resolución Espacial de Dos Fuentes Puntuales (Criterio de Rayleigh)', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


## 5. Redes de Difracción

Una **red de difracción** es un dispositivo óptico que consta de una superficie con una gran cantidad de ranuras paralelas y equidistantes separadas por una distancia $d$ (llamada período de la red). Se utilizan para dispersar la luz en sus diferentes colores constituyentes con un poder de resolución muy superior al de los prismas.

### 5.1 Ecuación de la Red de Difracción
Cuando un haz de luz incide sobre la red con un ángulo $\theta_i$ respecto a la normal, la luz se difracta en cada rendija. Las ondas difractadas interfieren de forma constructiva en la región de campo lejano únicamente para los ángulos $\theta_m$ donde la diferencia de camino óptico entre rendijas adyacentes es un múltiplo entero de la longitud de onda:
$$d (\sin\theta_m \pm \sin\theta_i) = m\lambda, \quad m \in \mathbb{Z}$$
Donde:
* $m$ es el **orden de difracción** (0, $\pm 1$, $\pm 2$, ...).
* Si la incidencia es perpendicular a la red ($\theta_i = 0$), la ecuación se simplifica a:
  $$d \sin\theta_m = m\lambda$$

### 5.2 Perfil de Intensidad para $N$ Rendijas
Considerando una red de $N$ rendijas de ancho individual $a$ y separación entre centros $d$, la intensidad resultante viene dada por el producto del factor de difracción de una sola rendija y el factor de interferencia de $N$ fuentes delgadas:
$$I(\theta) = I_0 \left[ \frac{\sin(\beta)}{\beta} \right]^2 \left[ \frac{\sin(N \alpha)}{N \sin(\alpha)} \right]^2$$
Donde:
$$\beta = \frac{\pi a \sin\theta}{\lambda}, \quad \alpha = \frac{\pi d \sin\theta}{\lambda}$$
A medida que $N$ aumenta, los máximos principales se vuelven extremadamente estrechos e intensos (con un ancho proporcional a $1/N$). Los máximos secundarios intermedios tienen intensidades despreciables, lo que permite discriminar longitudes de onda muy cercanas.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parámetros físicos de la red de difracción
d = 2.0e-6        # Período de la red (2 micrometros, correspondiente a 500 líneas/mm)
a = 0.45e-6       # Ancho de cada rendija (0.45 micrometros)
L = 1.0           # Distancia de la red a la pantalla (1.0 m)

# Grid angular en la pantalla (de -45 a 45 grados para capturar el primer orden)
y_scr = np.linspace(-0.6, 0.6, 1200)
theta = np.arctan(y_scr / L)

def grating_profile(theta, d, a, lam, N):
    alpha = np.pi * d * np.sin(theta) / lam
    beta = np.pi * a * np.sin(theta) / lam
    
    sin_alpha = np.sin(alpha)
    # Evitar divisiones por cero en los máximos de interferencia (donde sin_alpha -> 0)
    sin_alpha = np.where(np.abs(sin_alpha) < 1e-9, 1e-9, sin_alpha)
    
    interference = (np.sin(N * alpha) / (N * sin_alpha))**2
    diffraction = np.sinc(a * np.sin(theta) / lam)**2
    return interference * diffraction

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# 1. Subplot 1: Efecto del número de rendijas N (Luz monocromática de 532 nm - verde)
N_vals = [2, 5, 18]
lam_green = 532e-9
colors_N = ['blueviolet', 'forestgreen', 'darkorange']

for N, col in zip(N_vals, colors_N):
    I_val = grating_profile(theta, d, a, lam_green, N)
    ax1.plot(y_scr * 100, I_val, color=col, linewidth=1.8, label=f'$N = {N}$ rendijas')

ax1.set_title('Aguzamiento de los Máximos con $N$ Creciente', fontsize=12, fontweight='bold')
ax1.set_xlabel('Posición en pantalla $y$ (cm)', fontsize=11)
ax1.set_ylabel('Intensidad Relativa $I/I_0$', fontsize=11)
ax1.set_xlim(-45, 45)
ax1.grid(True, linestyle=':', alpha=0.5)
ax1.legend(frameon=True, loc='upper right')

# 2. Subplot 2: Dispersión cromática para N=10 (Separación de Rojo, Verde y Azul)
wavelengths_dict = {
    'Azul (450 nm)': (450e-9, 'blue'),
    'Verde (532 nm)': (532e-9, 'green'),
    'Rojo (650 nm)': (650e-9, 'red')
}
N_disp = 10

for label, (lam_val, color_line) in wavelengths_dict.items():
    I_disp = grating_profile(theta, d, a, lam_val, N_disp)
    ax2.plot(y_scr * 100, I_disp, color=color_line, linewidth=2.0, label=label)

ax2.set_title(r'Dispersión Cromática (Separación en Órdenes $m = \pm 1$)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Posición en pantalla $y$ (cm)', fontsize=11)
ax2.set_ylabel('Intensidad Relativa $I/I_0$', fontsize=11)
ax2.set_xlim(-45, 45)
ax2.grid(True, linestyle=':', alpha=0.5)
ax2.legend(frameon=True, loc='upper right')

plt.suptitle('Simulación Física de una Red de Difracción', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


## 6. Validación Numérica y Verificación de la Física

Para garantizar que los modelos matemáticos y simulaciones numéricas sean conformes con las leyes físicas, implementamos esta celda de validación usando aserciones (`assert`):
1. **Mínimos de la rendija única:** El primer mínimo teórico de difracción de Fraunhofer para una rendija de ancho $a$ debe coincidir exactamente con el ángulo donde la intensidad se hace cero.
2. **Cero del disco de Airy:** Comprobamos que el primer cero de la función de Bessel $J_1(u)$ ocurra a $u \approx 3.8317$, lo cual valida físicamente el factor $1.22$ del criterio de resolución de Rayleigh.
3. **Picos de la red de difracción:** Comprobamos que los máximos principales del primer orden $m=1$ de la red de difracción ocurran en el ángulo predicho por la ecuación de red $d \sin\theta = \lambda$.


In [ ]:
from scipy.special import j1
import numpy as np

# 1. Validación de los mínimos de la rendija única
a_width = 80e-6     # 80 micrometros
lam_val = 633e-9    # 633 nm

# Primer mínimo m=1: sin(theta) = lam / a
sin_theta_min_teorico = lam_val / a_width
theta_min_teorico = np.arcsin(sin_theta_min_teorico)

# Evaluar el parámetro beta en el mínimo teórico: beta = pi * a * sin(theta) / lam
beta_teorico = np.pi * a_width * np.sin(theta_min_teorico) / lam_val
print(f'Beta teórico en el primer mínimo (debe ser pi): {beta_teorico:.6f}')
assert np.isclose(beta_teorico, np.pi), 'Error: el parámetro beta no coincide con pi'

# 2. Validación del cero del Disco de Airy y factor 1.22
# El primer anillo oscuro ocurre cuando J_1(u) = 0
# Evaluamos u_cero = 3.83170597
u_cero = 3.83170597
j1_value_cero = j1(u_cero)
print(f'Valor de J_1(u) en u = 3.831706: {j1_value_cero:.6e}')
assert np.isclose(j1_value_cero, 0.0, atol=1e-6), 'u no es un cero preciso de la función de Bessel J_1'

# El factor es: u_cero / pi = 3.83170597 / 3.14159265 = 1.219669... ≈ 1.22
factor_Rayleigh = u_cero / np.pi
print(f'Factor teórico derivado de Rayleigh: {factor_Rayleigh:.4f}')
assert np.isclose(factor_Rayleigh, 1.22, atol=1e-2), 'Error en la derivación del factor de resolución 1.22'

# 3. Validación de los picos de la Red de Difracción
d_period = 2.0e-6   # Red de 500 lineas/mm
lam_light = 532e-9  # Luz verde (532 nm)

# Primer orden m=1: sin(theta_1) = lam / d
sin_theta_1_teorico = lam_light / d_period
theta_1_teorico = np.arcsin(sin_theta_1_teorico)

# El parámetro alfa es: alfa = pi * d * sin(theta) / lam
# En el pico de interferencia, sin(alfa) -> 0, lo cual ocurre en alfa = m * pi
alfa_teorico = np.pi * d_period * np.sin(theta_1_teorico) / lam_light
print(f'Alfa teórico en el primer orden de difracción (debe ser pi): {alfa_teorico:.6f}')
assert np.isclose(alfa_teorico, np.pi), 'Error: el máximo de la red no coincide con el orden m=1'

print('\n¡Todas las validaciones físicas y matemáticas del cuaderno de difracción se completaron con éxito!')


## 7. Bibliografía

1. *“Difracción de Fresnel”*, Universidad del País Vasco, disponible en: [ehu.es](http://www.sc.ehu.es/sbweb/fisica/ondas/fresnel/fresnel.htm).
2. Marqués, J. L., *“Difracción de Fraunhofer en una rendija”*, Universidad de la Rioja, 2002. Disponible en: [unirioja.es](https://www.unirioja.es/dptos/dq/fa/emo/amplia/node2.html).
3. Moreno, D; Gummy, S. *“Ojos para lo infinitesimal VI: buscando el límite”*, Principia. Disponible en: [principia.io](https://principia.io/2017/02/02/ojos-para-lo-infinitesimal-vi-buscando-el-limite.IjUxOCI/).
4. Ceressetto, M. V.; Grigera, T. S., *“Sobre el criterio de resolución de Rayleigh para fuentes policromáticas”*, CEILAP, 1991. Disponible en: [sedici.unlp.edu.ar](http://sedici.unlp.edu.ar/bitstream/handle/10915/142954/Documento_completo.pdf?sequence=1).
5. Pradillo, B. *“Redes de difracción”*, Orbitales moleculares, 2017. Disponible en: [orbitalesmoleculares.com](https://www.orbitalesmoleculares.com/redes-de-difraccion/).
